In [35]:
import math
import os
import time
import contextlib
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import rdchem
from torch import distributed as dist
from torch.utils.data import DataLoader, Dataset, DistributedSampler, Sampler

RDLogger.DisableLog("rdApp.*")

In [36]:
def _build_mapping(values: Iterable[int]) -> Dict[int, int]:
    """
    Creates a hashmap within which keys correspond to features and values correspond to integer encodings

    Args:
        values (Iterable[int]): atom/bond, i.e., node/edge features to encode

    Returns:
        Dict[int, int]: mapping between atom/ bond (node/ edge features) and their corresponding encodings
    """
    return {value: idx for idx, value in enumerate(values)}

def _one_hot(value: any, mapping: Dict[any,int]) -> np.ndarray[float]:
    """
    Create a one-hot encoded vector of floats (0.0) wherein the index of the value corresponding to the input feature reads 1.0.
    This index is given by the numerical encoding of the input feature and this numerical encoding is in turn given by an input hashmap.

    Args:
        value (any): input feature (e.g., atom type, atom degree, hybridization state)
        mapping (Dict[any:int]): a hashmap connecting the input feature to its numerical encoding

    Returns:
        np.ndarray: a vector of zeros (0.0, float) wherein the index corresponding to the numerical encoding of the input feature has been switched to 1.0 (float)
    """
    size = len(mapping) + 1 
    vec = np.zeros(size, dtype=np.float32) # use floats rather than integers for neural network training
    vec[mapping.get(value, len(mapping))] = 1.0 # flip the bit corresponding to the mapping index to 1
    return vec


In [ ]:
### ------ atom features used to create node matrix ------
ATOM_TYPES = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53] # [H, B, C, N, O, F, Si, P, S, Cl, Br, I]
ATOM_MAP = _build_mapping(ATOM_TYPES)

DEGREES = [0, 1, 2, 3, 4, 5]
DEGREE_MAP = _build_mapping(DEGREES)

FORMAL_CHARGES = [-2,1,0,1,2]
CHARGE_MAP = _build_mapping(FORMAL_CHARGES)

NUM_HS = [0, 1, 2, 3, 4]
NUM_H_MAP = _build_mapping(NUM_HS)

HYBRIDIZATIONS = [rdchem.HybridizationType.SP,
                  rdchem.HybridizationType.SP2,
                  rdchem.HybridizationType.SP3,
                  rdchem.HybridizationType.SP3D,
                  rdchem.HybridizationType.SP3D2]
HYB_MAP = _build_mapping(HYBRIDIZATIONS)

### ------ bond features used to create edge matrix ------
BOND_TYPES = [rdchem.BondType.SINGLE,
              rdchem.BondType.DOUBLE,
              rdchem.BondType.TRIPLE,
              rdchem.BondType.AROMATIC]
BOND_MAP = _build_mapping(BOND_TYPES)

In [43]:
# one-hot encoding for atomic number of carbon (hence value = 6)
_one_hot(value = 6, mapping = ATOM_MAP)

array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [45]:
def atom_to_feature(atom: rdchem.Atom) -> np.ndarray[float]:
    """
    Generates one-hot encoded vectors for each atom feature for a given RDKit rdchem.Atom object.
    These one-hot encoded feature vectors are then concatenated along the row-dimension to output a single row vector.

    Args:
        atom (rdchem.Atom): RDKit atom object.

    Returns:
        np.ndarray[float]: Single output vector formed by concatenating each one-hot-encoded feature vector along the row dimension.
    """
    feats = [
        _one_hot(atom.GetAtomicNum(), ATOM_MAP),
        _one_hot(atom.GetTotalDegree(), DEGREE_MAP),
        _one_hot(atom.GetFormalCharge(), CHARGE_MAP),
        _one_hot(atom.GetTotalNumHs(includeNeighbors=True), NUM_H_MAP),
        _one_hot(atom.GetHybridization(), HYB_MAP),
        np.array([atom.GetIsAromatic()], dtype=np.float32),
        np.array([atom.IsInRing()], dtype=np.float32),
    ]
    return np.concatenate(feats, axis=0)

In [46]:
mol = Chem.MolFromSmiles("CO")
for atom in mol.GetAtoms():
    print(f"Encoding for {atom.GetSymbol()} atom:")
    print(atom_to_feature(atom))
    print('')

Encoding for C atom:
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0.
 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]

Encoding for O atom:
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0.
 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]



In [ ]:
def bond_to_feature(bond: rdchem.Bond) -> np.ndarray[float]:
    vec = np.zeros()